In [8]:
import requests, pandas as pd

url = "https://gis.elpasotexas.gov/arcgis/rest/services/CID/VZ_Statistics/FeatureServer/4/query?where=1%3D1&outFields=*&returnGeometry=false&f=json"

# 🔹 Parameters to get *all attributes only*
params = {
    "where": "1=1",                      # all features
    "outFields": "*",                    # all fields
    "returnGeometry": "false",           # omit geometry
    "f": "json",                         # JSON response
    "resultRecordCount": 2000            # can adjust for large datasets
}

# 🔹 Query the service
response = requests.get(url, params=params)
data = response.json()

# 🔹 Extract attributes
records = [f["attributes"] for f in data["features"]]

# 🔹 Convert to a DataFrame
df = pd.DataFrame(records)

df

,OBJECTID,Day_of_Week,time_00_00,time_01_00,time_02_00,time_03_00,time_04_00,time_05_00,time_07_00,time_08_00,...,time_18_00,time_19_00,time_20_00,time_21_00,time_22_00,time_23_00,Severity,Year,time_06_00,time_09_00
0,1,Monday,0,3,1,0,0,0,0,0,...,0,0,1,0,1,2,K,2020,0,0
1,2,Tuesday,0,0,0,0,0,0,2,0,...,0,0,2,0,1,2,K,2020,0,0
2,3,Wednesday,0,1,0,1,2,2,0,0,...,0,5,0,4,3,1,K,2020,0,0
3,4,Thursday,0,0,0,0,0,0,0,0,...,2,0,4,1,0,0,K,2020,0,0
4,5,Friday,0,0,0,0,0,0,0,0,...,5,0,0,2,3,2,K,2020,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,80,Wednesday,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,A,2025,0,0
80,81,Thursday,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,A,2025,0,0
81,82,Friday,0,0,0,0,0,0,0,0,...,2,0,0,0,0,2,A,2025,0,0
82,83,Saturday,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,A,2025,0,0


In [6]:
data

{'objectIdFieldName': 'OBJECTID',
 'globalIdFieldName': '',
 'fields': [{'name': 'OBJECTID',
   'alias': 'OBJECTID',
   'type': 'esriFieldTypeOID'},
  {'name': 'C_Group',
   'alias': 'C_Group',
   'type': 'esriFieldTypeString',
   'length': 255},
  {'name': 'C_2020', 'alias': 'C_2020', 'type': 'esriFieldTypeBigInteger'},
  {'name': 'C_2021', 'alias': 'C_2021', 'type': 'esriFieldTypeBigInteger'},
  {'name': 'C_2022', 'alias': 'C_2022', 'type': 'esriFieldTypeBigInteger'},
  {'name': 'C_2023', 'alias': 'C_2023', 'type': 'esriFieldTypeBigInteger'},
  {'name': 'C_2024', 'alias': 'C_2024', 'type': 'esriFieldTypeBigInteger'},
  {'name': 'C_2025', 'alias': 'C_2025', 'type': 'esriFieldTypeBigInteger'}],
 'features': [{'attributes': {'OBJECTID': 1,
    'C_Group': 'Cyclist',
    'C_2020': 2,
    'C_2021': 7,
    'C_2022': 1,
    'C_2023': 6,
    'C_2024': 5,
    'C_2025': 1}},
  {'attributes': {'OBJECTID': 2,
    'C_Group': 'Motorcyclist',
    'C_2020': 14,
    'C_2021': 23,
    'C_2022': 26,
   

In [1]:
import requests

url = "https://gis.elpasotexas.gov/arcgis/rest/services/CID/2020_2025_EP_Collisions/FeatureServer/0/query"

params = {
    "where": "1=1",
    "returnGeometry": "false",
    "f": "json",
    "outStatistics": """[
        {"statisticType": "sum", "onStatisticField": "Death_Count", "outStatisticFieldName": "Total_Deaths"},
        {"statisticType": "sum", "onStatisticField": "Suspected_Serious_Injury_Count", "outStatisticFieldName": "Total_Serious_Injuries"}
    ]"""
}

response = requests.get(url, params=params).json()
print(response)


{'error': {'code': 500, 'message': 'Error performing query operation', 'details': []}}


In [105]:
import pandas as pd

ep_collision_data = pd.read_csv("C:/Users/CastroJG/Source/repos/SOMifold/data/2020_2025_CRIS.csv")

filtered_data = ep_collision_data[
    (ep_collision_data['On System Flag'] != 'Yes') &  # Not 'Yes'
    (ep_collision_data['On System Flag'].notna())     # Not NaN
]

# Optional: reset index
ep_collision_data = filtered_data.reset_index(drop=True)

C:\Users\CastroJG\AppData\Local\Temp\ipykernel_17892\1096487111.py:3: DtypeWarning: Columns (80,81,82,83) have mixed types. Specify dtype option on import or set low_memory=False.
  ep_collision_data = pd.read_csv("C:/Users/CastroJG/Source/repos/SOMifold/data/2020_2025_CRIS.csv")


In [106]:
len(ep_collision_data)

144486

In [55]:
# Get counts as a Series
person_type_counts = ep_collision_data['Person Type'].value_counts()

# Convert Series to DataFrame
person_type_counts = person_type_counts.reset_index()
person_type_counts.columns = ['Person Type', 'Count']

# Map each person type to a group
mapping = {
    '1 - DRIVER': 'MV',
    '2 - PASSENGER/OCCUPANT': 'MV',
    '4 - PEDESTRIAN': 'Pedestrian',
    '5 - DRIVER OF MOTORCYCLE TYPE VEHICLE': 'Motorcycle',
    '6 - PASSENGER/OCCUPANT ON MOTORCYCLE TYPE VEHICLE': 'Motorcycle',
    '3 - PEDALCYCLIST': 'Cyclist',
    '98 - OTHER (EXPLAIN IN NARRATIVE)': 'Other',
    '99 - UNKNOWN': 'Unknown'
}

person_type_counts['Group'] = person_type_counts['Person Type'].map(mapping)

# Sum counts by group
summary = person_type_counts.groupby('Group', as_index=False)['Count'].sum()

summary_person_type_counts = summary

In [56]:
severity_counts = ep_collision_data['Crash Severity'].value_counts()

# Convert to DataFrame
severity_df = severity_counts.reset_index()
severity_df.columns = ['Crash Severity', 'Count']

# Map original codes to new keys
severity_mapping = {
    'A - SUSPECTED SERIOUS INJURY': 'Serious Injury',
    'K - FATAL INJURY': 'Fatal Injury',
    'B - SUSPECTED MINOR INJURY': 'Minor Injury',
    'C - POSSIBLE INJURY': 'Minor Injury',
    'N - NOT INJURED': 'No Injury',
    '99 - UNKNOWN': 'Unknown'
}

# Apply mapping
severity_df['Group'] = severity_df['Crash Severity'].map(severity_mapping)

# Sum counts by new category
summary = severity_df.groupby('Group', as_index=False)['Count'].sum()
summary_severity_counts = summary

In [57]:
# Concatenate vertically (stack rows)
combined_df = pd.concat([summary_person_type_counts, summary_severity_counts], ignore_index=True)

In [58]:
# Calculate total rows
total_collisions = len(ep_collision_data)

# Create a DataFrame for the total row
total_row = pd.DataFrame({
    'Group': ['Total Collisions'],
    'Count': [total_collisions]
})

# Append to combined DataFrame
combined_df = pd.concat([combined_df, total_row], ignore_index=True)

In [59]:
combined_df.to_csv("C:/Users/CastroJG/Source/repos/SOMifold/data/summary_collision_data.csv", index=False)

In [97]:
ep_collision_data = ep_collision_data[
    ep_collision_data['Crash Severity'].isin(['K - FATAL INJURY', 'A - SUSPECTED SERIOUS INJURY'])
]

In [98]:
def save_collision_summary(df, output_csv="collision_summary.csv"):
    """
    Computes summary statistics for collisions and saves to a CSV.
    
    Parameters
    ----------
    df : pd.DataFrame
        The collisions DataFrame with columns 'Crash Severity' and 'Person Type'.
    output_csv : str
        Path/filename for the output CSV.
    """
    # Compute summary statistics
    total_collisions = len(df)
    
    fatal_injuries = (df["Crash Severity"] == "K - FATAL INJURY").sum()
    serious_injuries = (df["Crash Severity"] == "A - SUSPECTED SERIOUS INJURY").sum()
    motorcycle_collisions = df["Person Type"].isin([
        "5 - DRIVER OF MOTORCYCLE TYPE VEHICLE",
        "6 - PASSENGER/OCCUPANT ON MOTORCYCLE TYPE VEHICLE"
    ]).sum()
    bicycle_collisions = (df["Person Type"] == "3 - PEDALCYCLIST").sum()
    pedestrian_collisions = (df["Person Type"] == "4 - PEDESTRIAN").sum()
    
    # Create summary DataFrame
    summary_df = pd.DataFrame({
        "Statistic": [
            "Total Collisions",
            "Fatal Injuries",
            "Serious Injuries",
            "Motorcycle Collisions",
            "Bicycle Collisions",
            "Pedestrian Collisions"
        ],
        "Value": [
            total_collisions,
            fatal_injuries,
            serious_injuries,
            motorcycle_collisions,
            bicycle_collisions,
            pedestrian_collisions
        ]
    })
    
    # Save to CSV
    summary_df.to_csv(output_csv, index=False)
    print(f"✅ Collision summary saved to '{output_csv}'")
    
    return summary_df

summary_df = save_collision_summary(ep_collision_data, r"C:\Users\CastroJG\Source\repos\system_safety_ETL\collision_summary.csv")

✅ Collision summary saved to 'C:\Users\CastroJG\Source\repos\system_safety_ETL\collision_summary.csv'


In [104]:
import pandas as pd

# --------------------------
# Helper Functions
# --------------------------

def map_person_types(df, person_col='Person Type'):
    """
    Maps original Person Type values to new grouped categories.
    """
    mapping = {
        '1 - DRIVER': 'Motorist',
        '2 - PASSENGER/OCCUPANT': 'Motorist',
        '5 - DRIVER OF MOTORCYCLE TYPE VEHICLE': 'Motorcyclist',
        'Bycicle ': 'Motorcyclist',
        '3 - PEDALCYCLIST': 'Cyclist',
        '4 - PEDESTRIAN': 'Pedestrian',
        '98 - OTHER (EXPLAIN IN NARRATIVE)': 'Other',
        '99 - UNKNOWN': 'Unknown'
    }
    df['Group'] = df[person_col].map(mapping)
    return df

def create_person_type_pivot(df, value_col='Crash ID', year_col='Crash Year', person_col='Person Type'):
    """
    Creates a pivot table counting crashes by Person Type (or mapped group) per year.
    Returns a grouped DataFrame with new categories.
    """
    # Pivot: rows = Person Type, columns = Crash Year, values = count
    pivot_df = pd.pivot_table(
        df,
        index=person_col,
        columns=year_col,
        values=value_col,
        aggfunc='count',
        fill_value=0
    ).reset_index()
    
    # Map to grouped categories
    pivot_df = map_person_types(pivot_df, person_col)
    
    # Group by the new Group and sum the counts
    combined_df = pivot_df.drop(columns=person_col).groupby('Group', as_index=False).sum()
    
    return combined_df

# --------------------------
# Example usage
# --------------------------

person_type_summary = create_person_type_pivot(ep_collision_data)

# 🔹 Sum across all rows for each year (ignore 'Crash Year' and 'Group' columns)
totals = df.iloc[:, 2:].sum()

# 🔹 Create a new row for total
total_row = pd.DataFrame([["Total"] + totals.tolist()], columns=["Group"] + list(totals.index))

# 🔹 Append total row to df
df_total = pd.concat([df, total_row], ignore_index=True)

df_total
print(df_total)


   OBJECTID              Statistic  Value  Group
0       1.0       Total Collisions   1908    NaN
1       2.0         Fatal Injuries    330    NaN
2       3.0       Serious Injuries   1578    NaN
3       4.0  Motorcycle Collisions    173    NaN
4       5.0     Bicycle Collisions     22    NaN
5       6.0  Pedestrian Collisions    153    NaN
6       NaN                    NaN   4164  Total


In [82]:
import pandas as pd

# --------------------------
# Helper Functions
# --------------------------

def map_month_numbers_to_abbr(df, month_col='Crash Month'):
    """
    Converts numeric month values to three-letter abbreviations.
    """
    month_map = {
        1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
        7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
    }
    df[month_col] = df[month_col].map(month_map)
    return df

def generate_monthly_summary(df):
    """
    Generates a monthly summary pivot table for crashes with:
        - rows = month
        - columns = years
        - latest year sum
        - average across all years
    
    Returns:
        pd.DataFrame: monthly summary with month abbreviations
    """
    # Pivot: rows = month, columns = year, values = count of crashes
    monthly_pivot = pd.pivot_table(
        df,
        index='Crash Month',
        columns='Crash Year',
        values='Crash ID',
        aggfunc='count',
        fill_value=0
    )
    
    # Latest year
    latest_year = monthly_pivot.columns.max()
    monthly_pivot['Latest Year Sum'] = monthly_pivot[latest_year]
    
    # Average across all years
    monthly_pivot['Average Across Years'] = monthly_pivot.mean(axis=1)
    
    # Reset index for cleaner display
    monthly_summary = monthly_pivot.reset_index()
    
    # Map month numbers to abbreviations
    monthly_summary = map_month_numbers_to_abbr(monthly_summary, 'Crash Month')
    
    return monthly_summary

# --------------------------
# Example usage
# --------------------------

monthly_summary = generate_monthly_summary(ep_collision_data)

print(monthly_summary)


Crash Year Crash Month  2020  2021  2022  2023  2024  2025  Latest Year Sum  \
0                  Jan    26    16    16    62    19    20               20   
1                  Feb    23     5     5    52    24    39               39   
2                  Mar    11     8    45    44    15    24               24   
3                  Apr    12    13    21    96    13     3                3   
4                  May    12    42    36    58    32     0                0   
5                  Jun    22    29    36    60    21     0                0   
6                  Jul    16     6    28    58    35     0                0   
7                  Aug    26    14    41    76    20     0                0   
8                  Sep    26    19    30    52    24     0                0   
9                  Oct    20    26    35    80    27     0                0   
10                 Nov     5    34    27    44    17     0                0   
11                 Dec    20    27    24    60    31

In [83]:
import pandas as pd

# --------------------------
# Helper Functions
# --------------------------

def standardize_day_names(df, day_col='Day of Week'):
    """
    Standardizes Day of Week names (capitalizes first letter).
    """
    df[day_col] = df[day_col].str.capitalize()
    return df

def create_heatmap_pivot(df, index_col='Day of Week', column_col='Hour of Day', value_col='Crash Year'):
    """
    Creates a pivot table counting value_col for index_col vs column_col.
    """
    pivot = pd.pivot_table(
        df,
        index=index_col,
        columns=column_col,
        values=value_col,
        aggfunc='count',
        fill_value=0
    )
    
    # Sort days in calendar order if they are day names
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    if set(pivot.index).issubset(set(day_order)):
        pivot = pivot.reindex(day_order)
    
    return pivot

def generate_hour_day_heatmaps(df):
    """
    Generates a dictionary of heatmap DataFrames (Day of Week × Hour of Day) per year.
    
    Returns:
        dict: {year: pivot_table_df}
    """
    # Copy relevant columns
    df_heat = df[['Hour of Day', 'Day of Week', 'Crash Year']].copy()
    
    # Standardize Day of Week names
    df_heat = standardize_day_names(df_heat, 'Day of Week')
    
    # Get unique years
    years = df_heat['Crash Year'].unique()
    
    heatmaps = {}
    for year in years:
        df_year = df_heat[df_heat['Crash Year'] == year]
        pivot_df = create_heatmap_pivot(df_year)
        heatmaps[year] = pivot_df
    
    return heatmaps

# --------------------------
# Example usage
# --------------------------

heatmaps = generate_hour_day_heatmaps(ep_collision_data)

# Show heatmap for 2020
print("Heatmap for 2020")
print(heatmaps[2020])


Heatmap for 2020
Hour of Day  00:00 - 00:59  01:00 - 01:59  02:00 - 02:59  03:00 - 03:59  \
Day of Week                                                               
Monday                   0              3              1              0   
Tuesday                  0              0              2              2   
Wednesday                0              1              0              1   
Thursday                 0              0              0              0   
Friday                   0              0              0              0   
Saturday                 2              2              3              1   
Sunday                   4              0              0              0   

Hour of Day  04:00 - 04:59  05:00 - 05:59  07:00 - 07:59  08:00 - 08:59  \
Day of Week                                                               
Monday                   0              0              0              0   
Tuesday                  0              0              2              0   
Wednesd

In [84]:
import pandas as pd

# --------------------------
# Helper Functions
# --------------------------

def filter_valid_data(df):
    """
    Filters out rows with invalid Person Age, Ethnicity, or Gender.
    """
    df_filtered = df.copy()
    df_filtered = df_filtered[df_filtered['Person Age'] != 'No Data']
    df_filtered = df_filtered[df_filtered['Person Ethnicity'] != '99 - UNKNOWN']
    df_filtered = df_filtered[df_filtered['Person Gender'] != '99 - UNKNOWN']
    
    # Convert age to numeric
    df_filtered['Person Age'] = pd.to_numeric(df_filtered['Person Age'], errors='coerce')
    df_filtered = df_filtered.dropna(subset=['Person Age'])
    
    return df_filtered

def categorize_age(df, bins=None, labels=None):
    """
    Categorizes Person Age into age groups.
    """
    if bins is None:
        bins = [0, 17, 24, 34, 44, 54, 64, 74, 120]
    if labels is None:
        labels = ['0-17', '18-24', '25-34', '35-44', '45-54', '55-64', '65-74', '75+']
        
    df['Age Group'] = pd.cut(df['Person Age'], bins=bins, labels=labels, right=True)
    
    # Fill any missing categories
    df['Age Group'] = df['Age Group'].cat.add_categories('Unknown').fillna('Unknown')
    
    return df

def create_pivot(df, index_col, column_col, value_col='Person Age'):
    """
    Creates a pivot table counting value_col per index_col and column_col.
    """
    pivot = df.pivot_table(
        index=index_col,
        columns=column_col,
        values=value_col,
        aggfunc='count',
        fill_value=0
    ).reset_index()
    return pivot

# --------------------------
# Main Function
# --------------------------

def prepare_demographics_pivots(df):
    """
    Prepares pivot tables for stacked bar charts by Race/Ethnicity, Age, and Gender.
    Returns a dictionary of pivot tables.
    """
    # Copy only relevant columns
    df_demo = df[['Crash Year', 'Person Ethnicity', 'Person Age', 'Person Gender']].copy()
    
    # Filter invalid data
    df_demo = filter_valid_data(df_demo)
    
    # Categorize age into bins
    df_demo = categorize_age(df_demo)
    
    # Fill missing values for plotting
    df_demo['Person Ethnicity'] = df_demo['Person Ethnicity'].fillna('Unknown')
    df_demo['Person Gender'] = df_demo['Person Gender'].fillna('Unknown')
    
    # Create pivot tables
    race_pivot = create_pivot(df_demo, 'Crash Year', 'Person Ethnicity')
    age_pivot = create_pivot(df_demo, 'Crash Year', 'Age Group')
    sex_pivot = create_pivot(df_demo, 'Crash Year', 'Person Gender')
    
    return {
        'race': race_pivot,
        'age': age_pivot,
        'sex': sex_pivot
    }

# --------------------------
# Example usage
# --------------------------

demographics_pivots = prepare_demographics_pivots(ep_collision_data)

print("Race/Ethnicity pivot:")
print(demographics_pivots['race'].head())

print("\nAge pivot:")
print(demographics_pivots['age'].head())

print("\nSex pivot:")
print(demographics_pivots['sex'].head())


Race/Ethnicity pivot:
Person Ethnicity  Crash Year  98 - OTHER  A - ASIAN  B - BLACK  H - HISPANIC  \
0                       2020           0          1          6           157   
1                       2021           0          0         17           155   
2                       2022           5          1         14           241   
3                       2023           0          2         30           562   
4                       2024           2          1          9           202   

Person Ethnicity  I - AMER. INDIAN/ALASKAN NATIVE  No Data  W - WHITE  
0                                               0        0         42  
1                                               0        0         58  
2                                               0        5         65  
3                                               0        2        120  
4                                               0        2         52  

Age pivot:
Age Group  Crash Year  0-17  18-24  25-34  35-44  45-